# 03 — Experiment with model code directly

Run model code **inside this notebook's Python kernel**. Choose the model and
use `csv_input` from the shared `notebooks/config.json`, just as notebook 02 does.
The input may be an S3 URI or a local CSV path. Notebook 01 is not required when
the CSV already exists.

Read the model's existing parameters without editing them. Inspect a small
dataframe, then run selection → preparation → training → prediction →
post-processing. The notebook sets its environment variables explicitly before
importing model code. Restart the kernel when changing models or code that
loads settings at import time.

This notebook does not launch SageMaker jobs, register models or install
packages. Use the model's supported Python environment. Its local training
sample is for experimentation, not a held-out accuracy measurement.


## Choose the model and read the shared config

- Locate the checkout and load notebooks/config.json.
- Choose the model and use the existing csv_input from that config.
- Create a fresh local output folder for this experiment.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "mlops-models/src/mlops_models").is_dir())
sys.path[:0] = [str(ROOT / "mlops-models/src"), str(ROOT / "mlops-models")]

config_path = ROOT / "notebooks/config.json"
notebook_config = json.loads(config_path.read_text())
csv_input = notebook_config.get("csv_input", "").strip()
if not csv_input:
    raise ValueError(f"Set csv_input in {config_path} to the shared S3 URI or a local CSV.")

MODEL_ID = "usecase1"       # Choose your model folder.
TRAINING_MODE = "1"        # Preliminary training; your trainer owns the mode behavior.
MAX_ROWS = 200             # Keep the exploratory sample small.

MODEL_SOURCE = ROOT / "mlops-models/src/mlops_models/models" / MODEL_ID
RUN = ROOT / ".runs/model-experiments" / MODEL_ID / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")
RUN.mkdir(parents=True)
print("Kernel:", sys.executable)
print("Model source:", MODEL_SOURCE)
print("CSV:", csv_input)
print("Local outputs:", RUN)


## Set the notebook environment

- Set the model identity, training mode and CSV analysis mode before imports.
- Point model/output variables at local folders; the parameter folder is the model source until training finishes.
- Disable managed tracking and run reports for this local experiment.


In [ ]:
model_directory = RUN / "model"
training_reports = RUN / "training-reports"
model_directory.mkdir(exist_ok=True)
training_reports.mkdir(exist_ok=True)

# These values apply to this notebook kernel. No dotenv file is needed.
notebook_environment = {
    "MODEL_ID": MODEL_ID,
    "MLOPS_TRAINING_MODE": TRAINING_MODE,
    "MLOPS_MODEL_DIR": str(MODEL_SOURCE),  # Existing parameters, before a saved model is available.
    "SM_MODEL_DIR": str(model_directory),
    "SM_OUTPUT_DATA_DIR": str(training_reports),
    "SM_OUTPUT_DIR": str(training_reports),
    "MLOPS_ANALYSIS_MODE": "prediction_inference",
    "MLOPS_PREDICTION_FORMAT": "csv",
    "MLOPS_MODEL_METADATA_URI": "none",   # Preliminary training starts without prior metadata.
    "MLOPS_TRACKING_URI": "",
    "MLOPS_WRITE_RUN_REPORT": "0",
    "MLOPS_LOG_ARTIFACTS": "0",
    "TUNING_OBJECTIVE": "",
}
os.environ.update(notebook_environment)
for name, value in notebook_environment.items():
    print(name, "=", repr(value))


## Read the model parameters

- Read the selected model's existing parameters.json.
- Keep those settings unchanged and display them for reference.
- Start preliminary training without prior model metadata.


In [ ]:
import pandas as pd
from IPython.display import display

parameter_path = MODEL_SOURCE / "parameters.json"
parameters = json.loads(parameter_path.read_text())
metadata_input = {}
print("Parameters:", parameter_path)
display(parameters)


## Inspect a small dataframe

- Read the configured CSV with the notebook's AWS identity when it is on S3.
- Keep the ID column as text and load at most MAX_ROWS.
- Display the dataframe; this cell does not transform or save it.


In [ ]:
read_options = dict(nrows=MAX_ROWS, dtype={parameters["primary_key"]: str},
                    keep_default_na=False, na_values=["", r"\N"])
if csv_input.startswith("s3://"):
    import boto3
    bucket, key = csv_input.removeprefix("s3://").split("/", 1)
    with boto3.client("s3").get_object(Bucket=bucket, Key=key)["Body"] as stream:
        df = pd.read_csv(stream, **read_options)
else:
    csv_path = Path(csv_input).expanduser()
    if not csv_path.is_absolute():
        csv_path = ROOT / csv_path
    df = pd.read_csv(csv_path, **read_options)
print("CSV:", csv_input, "| rows / columns:", df.shape)
display(df.head())


## Import the code you want to experiment with

- Import the real base selector, preparation and training modules.
- Show their source files and the framework class behind ModelTraining.
- Keep these imports editable when testing a use-case override.

For example, import your use case's selector or trainer instead of the base
class when testing an override. After editing source, restart the kernel and
run from the top for a clean experiment. In VS Code, set a breakpoint in the
imported `.py` file and use **Debug Cell** to step into the actual function.

In [ ]:
from mlops_models.model_base.training import feature_selection, preprocess, train

for module in (feature_selection, preprocess, train):
    print(module.__name__, "->", module.__file__)
print("Training class:", train.ModelTraining.__module__, train.ModelTraining.__name__)

## Fit and inspect the selector

- Construct the real selector with the experiment settings.
- Fit it to the dataframe and retain the selector object for inspection.
- Export its selected-feature CSV and PDF through the existing output function.

In [ ]:
selector = feature_selection.FeatureSelector(parameters)
selector.fit(df)
selection_directory = feature_selection.write_container_output(selector, RUN)
selection_file = selection_directory / "selected_features.csv"
display(pd.read_csv(selection_file))
print("Selection outputs:", selection_directory)

## Run the model's training preparation

- Save the inspected dataframe locally because the preparation function accepts a directory containing data.csv.
- Pass that directory, selected features and the original parameter path to the model code.
- Inspect the prepared dataframe; the shared source CSV stays unchanged.


In [ ]:
raw_directory = RUN / "raw"
raw_directory.mkdir(exist_ok=True)
df.to_csv(raw_directory / "data.csv", index=False)
preprocess.process_train(raw_directory, selection_file, RUN, metadata_input, parameter_path)
prepared_directory = RUN / "prepared"
prepared = pd.read_csv(prepared_directory / "data.csv", dtype={parameters["primary_key"]: str})
print("Prepared shape:", prepared.shape)
display(prepared.head())


## Train directly and retain the model outputs

- Construct the trainer with the original settings and the model identity supplied by the application.
- Train using the local feature and data paths; keep the trainer and metadata visible.
- Let the existing packaging function save the model and its settings for inference.

This cell performs real training in the notebook environment. Run from the top
for a fresh experiment. Evaluation on this sample is not an independent test.


In [ ]:
# Supply the runtime model identity without changing parameters or its source file.
trainer = train.ModelTraining(
    selection_file, prepared_directory, metadata_input, {**parameters, "model_id": MODEL_ID})
checkpoint, model_metadata = trainer.train_model(RUN / "checkpoint", training_reports)
model_directory = train.write_container_output(
    checkpoint, model_metadata, model_directory, training_reports, parameters=parameters)
print("Saved model:", model_directory)
display(model_metadata)


## Set up prediction using the saved model

- Point the environment at this experiment's saved model and score output folder before inference imports.
- Read the parameters and metadata saved with that model.
- Run inference preparation on the local snapshot with the saved selection.


In [ ]:
score_directory = RUN / "scores"
score_directory.mkdir(exist_ok=True)
os.environ["MLOPS_MODEL_DIR"] = str(model_directory.resolve())
os.environ["SM_MODEL_DIR"] = str(model_directory.resolve())
os.environ["SM_OUTPUT_DATA_DIR"] = str(score_directory)
os.environ["SM_OUTPUT_DIR"] = str(score_directory)
saved_parameters = json.loads((model_directory / "parameters.json").read_text())
model_metadata = json.loads((model_directory / "model_metadata.json").read_text())
from mlops_models.model_base.inference import preprocess as inference_preprocess
from mlops_models.model_base.inference import predict

inference_preprocess.preprocess_data(
    raw_directory, selection_file, RUN / "inference", model_metadata,
    model_directory / "parameters.json")
inference_directory = RUN / "inference/prepared"
inference_frame = pd.read_csv(inference_directory / "data.csv",
                              dtype={saved_parameters["primary_key"]: str})
print("Model directory:", os.environ["MLOPS_MODEL_DIR"])
display(inference_frame.head())


## Inspect predictions and write the model's files

- Load the saved checkpoint through `gluonPredictor`, imported by the base as `ModelPredictor`.
- Keep a small prediction dataframe visible for experiments.
- Let the predictor write its normal CSV outputs into this experiment's score folder.

In [ ]:
predictor = predict.ModelPredictor(model_directory, model_metadata, saved_parameters)
print("Predictor class:", type(predictor).__module__, type(predictor).__name__)
predictions = predictor.predict_with_best_model(inference_frame.head(10))
display(predictions)
predictor.predict_file(
    inference_directory / "data.csv", model_metadata["feature_mapping"],
    saved_parameters["primary_key"], output_directory=score_directory)
predict.write_container_output(score_directory)


## Run the real post-processing function

- Set the analysis output folder before importing post-processing code.
- Pass the original snapshot, selection, scores and saved model to the analysis function.
- List its output files and preview prediction_analysis.csv when present.


In [ ]:
analysis_directory = RUN / "analysis"
analysis_directory.mkdir(exist_ok=True)
os.environ["SM_OUTPUT_DATA_DIR"] = str(analysis_directory)
os.environ["SM_OUTPUT_DIR"] = str(analysis_directory)
from mlops_models.model_base.inference import postprocess

postprocess.postprocess_predictions(
    raw_directory, selection_file, score_directory, analysis_directory, model_directory,
    analysis_mode=os.environ["MLOPS_ANALYSIS_MODE"])
for path in sorted(analysis_directory.rglob("*")):
    if path.is_file():
        print(path.relative_to(RUN))
analysis_csv = analysis_directory / "prediction_analysis.csv"
if analysis_csv.is_file():
    display(pd.read_csv(analysis_csv, dtype={saved_parameters["primary_key"]: str}).head())


## Continue the experiment

Inspect `df`, `selector`, `prepared`, `trainer`, `model_metadata`, `predictor`
and `predictions`. Edit the dataframe or the model code as needed, and keep
the model's algorithms and file writers.

Every run from the top gets a new local output folder. Restart the kernel after
changing models or module-level settings. Use **02** to publish reviewed code,
then **04** to start or inspect managed training/inference.
